# 06 — Lapse-history sensitivity analysis (Smoke Free)

Paper: Section 3.10, Table 11 and Supplement S5.

1. Run notebook 01 first: this notebook reads its results folder (`SAVE_DIR`) for the all-feature models.
2. Run the setup cell below. It executes notebook 01's setup cells (CELL 1–7) in this kernel, so the data, features, folds and model code are identical.
3. Run the analysis cell. Variant results go to `lapse_history_sensitivity/` (set `SENS_DIR_OVERRIDE` to change this). Finished folds are skipped after a restart.
4. To recompute Table 11 and Supplement S5 from the saved outputs, run `python scripts/analyze_lapse_history_sensitivity.py --main <SAVE_DIR> --sens lapse_history_sensitivity`.


In [ ]:
# Runs the setup cells of notebook 01 (CELL 1 to CELL 7: imports, data, features, models, CV engine,
# MODEL_CONFIGS) in this kernel, so that this notebook uses exactly the same code, folds and settings.
# Set QUICK_TEST / SAVE_DIR in notebook 01's first cell, or override them here after this cell runs.
import json, re, os
NB01 = '01_benchmark_24_configurations.ipynb'
_nb = json.load(open(NB01))
_setup = [c for c in _nb['cells'] if c['cell_type'] == 'code'
          and re.search(r'^# CELL (1|2|2b|3|4|5|6|7) —', ''.join(c['source']), re.M)]
assert len(_setup) == 8, f'expected 8 setup cells in {NB01}, found {len(_setup)}'
for _c in _setup:
    exec(compile(''.join(_c['source']), NB01, 'exec'), globals())
print('Notebook 01 setup cells run. SAVE_DIR =', SAVE_DIR)


In [ ]:
# ════════════════════════════════════════════════════════
# LAPSE-HISTORY SENSITIVITY ANALYSIS (paper: Section 3.10, Table 11, Supplement S5)
#
# Retrains all 24 configurations on the same participant folds as notebook 01, in two variants:
#   no_prior   : without prior_event_lapse
#   no_history : without prior_event_lapse, lapse_lag2, lapse_lag3, days_since_last_lapse
# For the original features the two variants coincide; the no_history results are copied from no_prior.
#
# Settings used in the paper (REUSE_MAIN_SETTINGS = False):
#   - traditional models: the notebook-01 grid search (ML_GRIDS) repeated on the same inner validation split;
#   - sequence models and self-supervised heads: learning rate and dropout fixed at DL_DEFAULT (the first
#     value of DL_GRID: lr 1e-3, dropout 0.3); only the number of epochs is chosen by early stopping on the
#     inner validation split (PATIENCE = 5, at most FT_EPOCHS = 30). Notebook 01 also searched lr and dropout;
#   - self-supervised encoders are pretrained from scratch in each fold on the reduced inputs (training
#     participants of that fold only) and then frozen; no encoder from the main run is reused.
# Finished folds are saved after each fold and skipped after a restart.
# ════════════════════════════════════════════════════════
import os, ast, glob, copy, time, json, shutil
import numpy as np, pandas as pd, torch
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from scipy import stats

# ── settings ────────────────────────────────────────────
try:                                                     # mount Google Drive only when running in Colab
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'): drive.mount('/content/drive')
except ImportError:
    pass
DRIVE_ROOT = '/content/drive/MyDrive' if os.path.exists('/content/drive/MyDrive') else None
MAIN_DIR_OVERRIDE = None   # folder with notebook-01 results (RF_exp.json, TFM_exp.json, ...); None = search near SAVE_DIR
REUSE_MAIN_SETTINGS = False   # False reproduces the paper (re-tuned as described above); True reuses notebook 01's per-fold settings
SENS_DIR_OVERRIDE = None   # leave None in Colab; outside Colab, the path of your lapse_history_sensitivity folder

def _find_main_dir():
    """Folder holding the main (tuned) benchmark results: <model>.json files with 'fold_aucs'."""
    names = [c['name'] for c in MODEL_CONFIGS]
    def score(d):
        n = 0
        for nm in names:
            f = os.path.join(d, f'{nm}.json')
            try:
                if os.path.exists(f) and 'fold_aucs' in json.load(open(f)): n += 1
            except Exception: pass
        return n
    if MAIN_DIR_OVERRIDE and os.path.isdir(MAIN_DIR_OVERRIDE):
        sc = score(MAIN_DIR_OVERRIDE)
        print(f'  main-results folder (override): {MAIN_DIR_OVERRIDE}  ({sc}/{len(names)} model files)')
        if sc: return MAIN_DIR_OVERRIDE
        print('  ⚠️ the override folder has no model .json files; searching elsewhere')
    cands = []
    if 'SAVE_DIR' in globals(): cands.append(SAVE_DIR)
    roots = [r for r in (DRIVE_ROOT, os.getcwd(), os.path.dirname(os.path.abspath(SAVE_DIR)) if 'SAVE_DIR' in globals() else None) if r]
    for root in roots:
        for depth in ('*', '*/*', '*/*/*'):
            cands += sorted({os.path.dirname(f) for f in glob.glob(os.path.join(root, depth, 'RF_exp.json'))})
    cands = [c for c in dict.fromkeys(cands) if c and 'lapse_history_sensitivity' not in c]
    scored = sorted(((score(c), c) for c in cands), reverse=True)
    for sc, c in scored[:5]: print(f'  candidate main-results folder: {c}  ({sc}/{len(names)} model files)')
    if not scored or scored[0][0] == 0:
        print('  ⚠️ main benchmark results NOT found: the "full" columns and P values will be empty.\n'
              '     Set MAIN_DIR_OVERRIDE to the folder that contains RF_exp.json, TFM_exp.json, ... and run again.')
        return SAVE_DIR if 'SAVE_DIR' in globals() else '.'
    return scored[0][1]
SENS_DIR = SENS_DIR_OVERRIDE or ((DRIVE_ROOT or '.') + '/lapse_history_sensitivity')
VARIANTS = {'no_prior':   ['prior_event_lapse'],
            'no_history': ['prior_event_lapse', 'lapse_lag2', 'lapse_lag3', 'days_since_last_lapse']}
RUN_VARIANTS = ['no_prior', 'no_history']   # finished folds are skipped, so after a complete run this cell only prints results
RUN_MODELS   = None                         # None = all 24; or e.g. ['RF_exp', 'RF_orig', 'GRUD_exp']
SHOW_RESULTS = True                         # print the results table at the end (also works while runs are incomplete)

for _v in ['df', 'MODEL_CONFIGS', 'build_sequences', 'fit_ml', 'train_dl', 'ssl_pretrain_fold', 'N_FOLDS', 'SEED']:
    if _v not in globals():
        raise RuntimeError(f"'{_v}' is not defined: run the setup cell of this notebook first "
                           "(after a disconnect all variables are lost), then run this cell again.")
MAIN_DIR = _find_main_dir()
OUT = SENS_DIR + ('_quicktest' if globals().get('QUICK_TEST') else '')
os.makedirs(OUT, exist_ok=True)
print(f'main results: {MAIN_DIR}\nthis run    : {OUT}\ndevice      : {device}')

# ── helpers ─────────────────────────────────────────────
def _atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f: json.dump(obj, f, default=str)
    os.replace(tmp, path)

def _atomic_csv(d_, path):
    tmp = path + '.tmp'; d_.to_csv(tmp, index=False); os.replace(tmp, path)

def _parse_hp(h):
    if isinstance(h, dict): return h
    if isinstance(h, str):
        for f in (ast.literal_eval, lambda s: json.loads(s.replace("'", '"'))):
            try: return f(h)
            except Exception: pass
    return None

def main_hparams(name):
    p = os.path.join(MAIN_DIR, f'{name}.json')
    if not os.path.exists(p): return None
    hp = json.load(open(p)).get('hparams', {})
    out = {t: [_parse_hp(h) for h in hp.get(t, [])] for t in ('classify', 'next')}
    return out if all(len(v) >= N_FOLDS for v in out.values()) else None

def vdir(variant):
    d = os.path.join(OUT, variant)
    os.makedirs(os.path.join(d, 'predictions'), exist_ok=True); os.makedirs(os.path.join(d, 'ssl_cache'), exist_ok=True)
    return d

def fold_done(variant, name, k):
    d = vdir(variant); p = os.path.join(d, f'{name}.json')
    if not os.path.exists(p) or str(k) not in json.load(open(p)).get('folds', {}): return False
    return all(os.path.exists(os.path.join(d, 'predictions', f'{name}__{t}__fold{k:02d}.csv')) for t in ('classify', 'next'))

def save_fold(variant, name, k, rec, meta):
    p = os.path.join(vdir(variant), f'{name}.json')
    allrec = json.load(open(p)) if os.path.exists(p) else {'folds': {}}
    allrec['folds'][str(k)] = rec; allrec.update(meta); _atomic_json(allrec, p)

def save_fold_preds(variant, name, task, k, pids, y, p):
    _atomic_csv(pd.DataFrame({'participant_id': np.asarray(pids, dtype=object), 'y_true': np.asarray(y, float).astype(int),
                              'y_prob': np.asarray(p, float)}),
                os.path.join(vdir(variant), 'predictions', f'{name}__{task}__fold{k:02d}.csv'))

# ── one configuration under one variant, fold by fold ───
def run_variant(cfg, variant):
    name, mtype = cfg['name'], cfg['model_type']
    drop = set(VARIANTS[variant])
    feat_cols = [c for c in cfg['feat_cols'] if c not in drop]
    removed = [c for c in cfg['feat_cols'] if c in drop]
    # same feature set already finished under the other variant (original features: no_history = no_prior) -> copy
    for other in RUN_VARIANTS + [v for v in VARIANTS if v not in RUN_VARIANTS]:
        if other == variant or set(c for c in cfg['feat_cols'] if c in set(VARIANTS[other])) != set(removed): continue
        if all(fold_done(other, name, k) for k in range(N_FOLDS)) and not all(fold_done(variant, name, k) for k in range(N_FOLDS)):
            src, dst = vdir(other), vdir(variant)
            for f in glob.glob(os.path.join(src, 'predictions', f'{name}__*.csv')):
                shutil.copy(f, os.path.join(dst, 'predictions', os.path.basename(f)))
            rec = json.load(open(os.path.join(src, f'{name}.json'))); rec.update(variant=variant, copied_from=other)
            _atomic_json(rec, os.path.join(dst, f'{name}.json'))
            print(f'  ↪ {variant:10s} {name:14s} same features as {other}: results copied'); return
    todo = [k for k in range(N_FOLDS) if not fold_done(variant, name, k)]
    if not todo:
        print(f'  ✅ {variant:10s} {name:14s} complete'); return
    nf = len(feat_cols)
    is_grud = mtype in ('grud', 'ssl_grud'); is_ml = mtype in ('rf', 'xgb', 'lr', 'svm'); is_ssl = mtype.startswith('ssl_')
    all_seqs = build_sequences_grud(df, feat_cols) if is_grud else build_sequences(df, feat_cols)
    inp_nf = nf * 3 if is_grud else nf
    pid_arr = np.array([s[2] for s in all_seqs])
    splits = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(pid_arr))   # same folds as the main run
    HP = main_hparams(name) if REUSE_MAIN_SETTINGS else None
    meta = {'model': name, 'variant': variant, 'n_features': nf, 'removed': removed,
            'settings_from': 'main run (per fold)' if HP else 're-tuned (see header)'}
    print(f'\n  ▶ {variant} | {name} | {nf} features (removed {removed or "none"}) | {meta["settings_from"]} | folds left {len(todo)}')
    t0 = time.time()
    for k in todo:
        tr_idx, te_idx = splits[k]
        set_seed(SEED + k)
        train_raw = [all_seqs[i] for i in tr_idx]; test_raw = [all_seqs[i] for i in te_idx]
        train_s, test_s, _, _ = standardize_fold(train_raw, test_raw, nf)
        perm = np.random.default_rng(SEED + k).permutation(len(train_raw))
        n_val = max(1, int(round(INNER_VAL_FRAC * len(train_raw))))
        itr_s, ival_s, _, _ = standardize_fold([train_raw[i] for i in perm[n_val:]], [train_raw[i] for i in perm[:n_val]], nf)
        ssl_enc = None
        if is_ssl:
            enc_cls = {'ssl_gru': GRUEncoder, 'ssl_lstm': LSTMEncoder, 'ssl_grud': GRUDEncoder}[mtype]
            cache = os.path.join(vdir(variant), 'ssl_cache', f'{name}_fold{k:02d}.pt')
            if os.path.exists(cache):
                ssl_enc = enc_cls(nf, hid=64, drop=0.3, n_layers=2).to(device)
                ssl_enc.load_state_dict(torch.load(cache, map_location=device))
            else:
                ssl_enc = ssl_pretrain_fold(train_s, enc_cls, nf, enc_kwargs={'hid': 64, 'drop': 0.3, 'n_layers': 2},
                                            epochs=SSL_EPOCHS, bs=8, lr=1e-3, wd=1e-4)
                torch.save(ssl_enc.state_dict(), cache + '.tmp'); os.replace(cache + '.tmp', cache)
        rec = {}
        for task in ['classify', 'next']:
            hp = HP[task][k] if HP else None
            if is_ml:
                if hp is None:                                   # repeat the main run's grid search on the inner split
                    grid = ML_GRIDS[mtype]; hp = grid[0]
                    Xi, yi, _ = flat(itr_s, task); Xv, yv, _ = flat(ival_s, task)
                    sc = [safe_auc(yv, fit_ml(mtype, h, Xi, yi)(Xv)) for h in grid]
                    if not np.all(np.isnan(sc)): hp = grid[int(np.nanargmax(sc))]
                Xtr, ytr, _ = flat(train_s, task); Xte, labels, pids = flat(test_s, task)
                probs = fit_ml(mtype, hp, Xtr, ytr)(Xte)
            else:
                def fresh(h):
                    set_seed(SEED + k)
                    enc = (copy.deepcopy(ssl_enc) if is_ssl else build_encoder(mtype, inp_nf, nf, h['drop'])).to(device)
                    if is_ssl:
                        for p_ in enc.parameters(): p_.requires_grad = False
                    return enc, PredHead(64, drop=h['drop']).to(device)
                if hp is None or 'epochs' not in hp:             # early stopping on the inner split with the default setting
                    h0 = dict(DL_DEFAULT) if hp is None else dict(hp)
                    enc, head = fresh(h0)
                    _, ep = train_dl(enc, head, is_ssl, itr_s, task, FT_EPOCHS, h0['lr'], 1e-4, 32, val_s=ival_s, patience=PATIENCE)
                    hp = dict(h0, epochs=int(ep))
                if globals().get('QUICK_TEST'): hp = dict(hp, epochs=1)
                enc, head = fresh(hp)
                train_dl(enc, head, is_ssl, train_s, task, int(hp['epochs']), hp['lr'], 1e-4, 32)
                probs, labels, pids = predict_dl(enc, head, test_s, task)
            save_fold_preds(variant, name, task, k, pids, labels, probs)
            rec[task] = safe_auc(labels, probs); rec[f'hp_{task}'] = hp
        save_fold(variant, name, k, rec, meta)
        print(f'    fold {k+1:2d}/{N_FOLDS}  classify={rec["classify"]:.4f}  next={rec["next"]:.4f}  ({(time.time()-t0)/60:.1f} min)')
        if device.type == 'cuda': torch.cuda.empty_cache()

def ordered_configs():
    cfgs = [c for c in MODEL_CONFIGS if RUN_MODELS is None or c['name'] in RUN_MODELS]
    return sorted(cfgs, key=lambda c: (c['model_type'] not in ('rf', 'xgb', 'lr', 'svm'), c['model_type'].startswith('ssl_')))

# ── run (traditional models first) ──────────────────────
t_all = time.time()
for variant in RUN_VARIANTS:
    for cfg in ordered_configs():
        run_variant(cfg, variant)
print(f'\n✅ requested runs complete ({(time.time() - t_all) / 60:.1f} min this session)')

# ── status ──────────────────────────────────────────────
st = pd.DataFrame([{'variant': v, 'model': c['name'], 'folds': sum(fold_done(v, c['name'], k) for k in range(N_FOLDS))}
                   for v in VARIANTS for c in ordered_configs()]).pivot(index='model', columns='variant', values='folds')
print('\nFolds finished (of', N_FOLDS, '):'); print(st.to_string())

# ── where the variant settings came from (report this in the Methods) ──
src = {}
for v in VARIANTS:
    for c in ordered_configs():
        f = os.path.join(OUT, v, f"{c['name']}.json")
        if os.path.exists(f): src[json.load(open(f)).get('settings_from', 'unknown')] = src.get(json.load(open(f)).get('settings_from', 'unknown'), 0) + 1
print('\nSettings used for the variant models:', src)

# ── results: full model vs no_prior vs no_history ───────
if SHOW_RESULTS:
    TASKS = {'classify': 'Classification', 'next': 'Next-entry'}
    LAB = {'lr': 'Logistic regression', 'rf': 'Random forest', 'xgb': 'XGBoost', 'svm': 'SVM', 'lstm': 'LSTM', 'gru': 'GRU',
           'grud': 'GRU-D', 'tcn': 'TCN', 'gru_tcn': 'GRU-TCN', 'transformer': 'Transformer', 'ssl_gru': 'SSL-GRU', 'ssl_grud': 'SSL-GRU-D'}
    def _folds(path, sens):
        if not os.path.exists(path): return None
        d = json.load(open(path))
        if not sens: return {t: (d.get('fold_aucs', {}).get(t, []) + [np.nan] * N_FOLDS)[:N_FOLDS] for t in TASKS}
        f = d.get('folds', {}); return {t: [f[str(k)][t] if str(k) in f else np.nan for k in range(N_FOLDS)] for t in TASKS}
    def _pp(pred_dir, name, task):
        fs = sorted(glob.glob(os.path.join(pred_dir, f'{name}__{task}__fold*.csv')))
        if len(fs) < N_FOLDS: return np.nan
        P = pd.concat([pd.read_csv(f, dtype={'participant_id': str}) for f in fs])
        a = [roc_auc_score(g.y_true, g.y_prob) for _, g in P.groupby('participant_id') if 0 < g.y_true.sum() < len(g)]
        return float(np.median(a)) if a else np.nan
    def _holm(p):
        p = np.asarray(p, float); out = np.full(len(p), np.nan); ok = ~np.isnan(p); q = p[ok]; o = np.argsort(q)
        adj = np.empty(len(q)); r_ = 0
        for r, i in enumerate(o): r_ = max(r_, min(1, (len(q) - r) * q[i])); adj[i] = r_
        out[ok] = adj; return out
    rows = []
    for c in ordered_configs():
        name = c['name']; full = _folds(os.path.join(MAIN_DIR, f'{name}.json'), False)
        for task, tl in TASKS.items():
            f = np.asarray(full[task], float) if full else np.full(N_FOLDS, np.nan)
            r = {'model': LAB[c['model_type']], 'features': 'original + engineered' if name.endswith('_exp') else 'original',
                 'task': tl, 'full': np.nanmean(f), 'full_pp': _pp(os.path.join(MAIN_DIR, 'predictions'), name, task)}
            for v in VARIANTS:
                va = _folds(os.path.join(OUT, v, f'{name}.json'), True)
                a = np.asarray(va[task], float) if va else np.full(N_FOLDS, np.nan)
                dd = a - f; ok = ~np.isnan(dd)
                r[v] = np.nanmean(a) if (~np.isnan(a)).any() else np.nan
                r[f'{v}_delta'] = dd[ok].mean() if ok.any() else np.nan
                r[f'{v}_p'] = stats.wilcoxon(dd[ok], method='exact').pvalue if ok.sum() >= 5 and np.any(dd[ok] != 0) else np.nan
                r[f'{v}_pp'] = _pp(os.path.join(OUT, v, 'predictions'), name, task)
            rows.append(r)
    R = pd.DataFrame(rows)
    for v in VARIANTS:
        for tl in TASKS.values():
            m = R.task == tl; R.loc[m, f'{v}_p_holm'] = _holm(R.loc[m, f'{v}_p'])
    R.to_csv(os.path.join(OUT, 'sensitivity_full_results.csv'), index=False)
    pd.set_option('display.width', 250); pd.set_option('display.float_format', '{:.3f}'.format)
    for tl in TASKS.values():
        cols = ['model', 'features', 'full'] + sum([[v, f'{v}_delta', f'{v}_p_holm'] for v in VARIANTS], []) + \
               ['full_pp'] + [f'{v}_pp' for v in VARIANTS]
        print(f'\n=== {tl}: pooled AUC (fold mean), change vs full model, median per-participant AUC (pp) ===')
        print(R[R.task == tl][cols].sort_values(['features', 'full'], ascending=[True, False]).to_string(index=False))
    print('\n=== Do the main conclusions hold? ===')
    for v in ['full'] + list(VARIANTS):
        for tl in TASKS.values():
            s = R[R.task == tl][v].dropna().sort_values(ascending=False)
            if len(s) >= 15: print(f'{v:10s} {tl:15s} best {s.iloc[0]:.3f}; 15 best within {s.iloc[0] - s.iloc[14]:.3f}')
            sub = R[R.task == tl].pivot(index='model', columns='features', values=v)
            if {'original', 'original + engineered'} <= set(sub.columns):
                g = (sub['original + engineered'] - sub['original']).dropna()
                tr = g[g.index.isin(['Logistic regression', 'Random forest', 'XGBoost', 'SVM'])]; sq = g.drop(tr.index)
                if len(tr) and len(sq):
                    print(f'{"":10s} {"":15s} engineered-feature gain: traditional {tr.min():+.3f} to {tr.max():+.3f}; '
                          f'sequence {sq.min():+.3f} to {sq.max():+.3f}')
            ppcol = 'full_pp' if v == 'full' else f'{v}_pp'
            gap = (R[R.task == tl][v] - R[R.task == tl][ppcol]).dropna()
            if len(gap): print(f'{"":10s} {"":15s} pooled − per-participant median: {gap.min():.3f} to {gap.max():.3f}')
    R[['model', 'features', 'task', 'full'] + list(VARIANTS) + ['full_pp'] + [f'{v}_pp' for v in VARIANTS]].to_csv(
        os.path.join(OUT, 'table_S5_lapse_history_sensitivity.csv'), index=False)
    print('\nSaved:', os.path.join(OUT, 'sensitivity_full_results.csv'), '+ table_S5_lapse_history_sensitivity.csv')
